# 02 — Metacell Coupling

Compute per-cell-type Spearman ρ between NR1I2 and each PXR target gene using a metacell aggregation strategy.

**Input**: `data/raw/nr1i2_atlas.h5ad`  
**Output**: `data/processed/coupling.csv`

### Method

1. For each cell type, aggregate cells into metacells via k-means (k = n_cells / 30) in log1p-PCA space.
2. Compute Spearman ρ between the mean NR1I2 profile and each target gene profile across metacells.
3. Cell types with fewer than `MIN_METACELLS=20` are skipped.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pxr_uncoupling.config import (
    COLOR_ACCENT, COLOR_CREAM,
    DATA_PROCESSED, DATA_RAW,
    MIN_METACELLS, NR1I2_SYMBOL, TARGET_CELLS_PER_METACELL,
)
from pxr_uncoupling.coupling import build_metacells, coupling_per_cell_type

## Load atlas

In [ ]:
adata = ad.read_h5ad(DATA_RAW / "nr1i2_atlas.h5ad")
print(f"{adata.n_obs:,} cells  ×  {adata.n_vars} genes")
target_genes = [g for g in adata.var_names if g != NR1I2_SYMBOL]
print(f"Target genes ({len(target_genes)}): {target_genes}")

## Metacell construction — worked example (hepatocyte)

Inspect one cell type to validate the metacell aggregation step.

In [ ]:
hep = adata[adata.obs["cell_type"] == "hepatocyte"]
print(f"Hepatocyte cells: {hep.n_obs}")

meta = build_metacells(hep, cells_per_metacell=TARGET_CELLS_PER_METACELL)
print(f"Metacells formed: {len(meta)}")
print(f"Mean cells/metacell: {hep.n_obs / len(meta):.1f}")
display(meta[[NR1I2_SYMBOL] + target_genes[:5]].describe().round(3))

In [ ]:
# Scatter: NR1I2 vs CYP3A4 across metacells
from scipy.stats import spearmanr

gene = "CYP3A4"
if gene in meta.columns:
    rho, p = spearmanr(meta[NR1I2_SYMBOL], meta[gene])

    fig, ax = plt.subplots(figsize=(5, 4))
    fig.patch.set_facecolor(COLOR_CREAM)
    ax.set_facecolor(COLOR_CREAM)
    ax.scatter(meta[NR1I2_SYMBOL], meta[gene], alpha=0.6, color=COLOR_ACCENT, edgecolors="none", s=30)
    ax.set_xlabel(f"{NR1I2_SYMBOL} (log1p mean)")
    ax.set_ylabel(f"{gene} (log1p mean)")
    ax.set_title(f"Hepatocyte metacells\nSpearman ρ = {rho:.3f}  (p={p:.2e})")
    plt.tight_layout()
    plt.show()

## Compute coupling for all cell types

In [ ]:
coupling = coupling_per_cell_type(
    adata,
    target_genes=target_genes,
    cells_per_metacell=TARGET_CELLS_PER_METACELL,
    min_metacells=MIN_METACELLS,
)
print(f"Coupling matrix: {coupling.shape}  ({coupling.shape[0]} cell types × {coupling.shape[1]} genes)")
display(coupling.round(3))

## Save

In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
out = DATA_PROCESSED / "coupling.csv"
coupling.to_csv(out)
print(f"Saved → {out}")

## Distribution of ρ values

In [ ]:
all_rho = coupling.values.flatten()
all_rho = all_rho[~np.isnan(all_rho)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor(COLOR_CREAM)
for ax in axes:
    ax.set_facecolor(COLOR_CREAM)

axes[0].hist(all_rho, bins=30, color=COLOR_ACCENT, edgecolor="#d0c4b0")
axes[0].axvline(0, color="#555", lw=1, ls="--")
axes[0].set_xlabel("Spearman ρ")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of all ρ values")

# Mean ρ per cell type
mean_rho = coupling.mean(axis=1).sort_values()
mean_rho.plot.barh(ax=axes[1], color=COLOR_ACCENT, edgecolor="#d0c4b0")
axes[1].axvline(0, color="#555", lw=1, ls="--")
axes[1].set_xlabel("Mean Spearman ρ (all target genes)")
axes[1].set_title("Mean coupling per cell type")

plt.tight_layout()
plt.show()

## Top coupled gene per cell type

In [ ]:
top = coupling.idxmax(axis=1).rename("top_gene").to_frame()
top["max_rho"] = coupling.max(axis=1)
display(top.sort_values("max_rho", ascending=False).round(3))